![image.png](https://i.imgur.com/a3uAqnb.png)

# Prompt Engineering, Measured: Voice-Assistant Intents in English and Arabic

- **Dataset**: [Amazon MASSIVE](https://huggingface.co/datasets/mteb/amazon_massive_intent) from Hugging Face, requests people said to a voice assistant, each labelled with an intent; the same requests exist in English and Arabic. We use 10 intents. [Natural Questions (open)](https://huggingface.co/datasets/google-research-datasets/nq_open) and [IMDb](https://huggingface.co/datasets/stanfordnlp/imdb) supply questions and long texts for the last two sections.
- **Task**: label each request with one intent, and measure how each change to the prompt moves the score
- **Model**: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), an instruction-tuned language model, used as it is (nothing is trained)
- **Runtime**: Colab T4 GPU, about 10 minutes of compute

In this lab, we will:

- score any prompt on the same labelled requests, in two languages
- describe the labels, choose examples per request, and reword the instruction, measuring each change
- use a persona to change who an answer is written for, and measure its reading level
- split a long text into pieces, one prompt per piece, and compare with a single long prompt

## Measuring a prompt in brief

A prompt change is judged like a model change: on labelled data, with one number per version. For classification the number is the accuracy:

$$\text{accuracy} = \frac{\text{requests labelled correctly}}{\text{requests in the test set}}$$

Where:
- a request counts as correct only if the reply contains the right intent name; a reply with no valid intent name counts as wrong
- every version of the prompt is scored on the same requests, decoded greedily (always the most probable token), so the score changes only when the prompt changes

With 100 requests per language, one request is one point: a gap of two or three points can be chance.

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. Everything this lab needs ships with Colab.

In [ ]:
import collections
import gzip
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # hide informational warnings from model loading and generation

Every size and model name lives in the next cell. `N_PER_INTENT` sets how many test requests each intent contributes per language, `N_QUESTIONS` how many questions the persona section asks, and `N_DOCS` how many long texts the last section builds.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-0.6B"     # try "Qwen/Qwen3-1.7B" in the exercises
INTENTS = ["alarm_set", "weather_query", "play_music", "calendar_set", "news_query", "email_sendemail",
           "cooking_recipe", "transport_query", "datetime_query", "iot_hue_lightoff"]
N_PER_INTENT = 10                # test requests per intent and language: 10 x 10 = 100 per language
N_EXAMPLES = 10                  # labelled examples shown in the few-shot prompts
MAX_NEW_TOKENS = 12              # an intent name is a few tokens
BATCH_SIZE = 16
N_QUESTIONS = 20                 # questions for the persona section
N_DOCS = 6                       # long texts per position in the last section
DOC_TOKENS = 6000                # length of each long text, in tokens
CHUNK_TOKENS = 500               # piece size when the long text is split

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so generation will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

## 1. Dataset

MASSIVE keeps each language in its own file, with an `id` shared by translations of the same request. We take `N_PER_INTENT` test requests per intent and the same ids in Arabic, so both languages are tested on the same requests. Labelled examples come from the training split.

In [ ]:
def massive(split, lang):
    path = hf_hub_download("mteb/amazon_massive_intent", f"{split}/{lang}.json.gz", repo_type="dataset")
    return [json.loads(line) for line in gzip.open(path, "rt")]

test_all = {lang: massive("test", lang) for lang in ["en", "ar"]}
train_all = {lang: massive("train", lang) for lang in ["en", "ar"]}

rng = random.Random(SEED)
by_intent = collections.defaultdict(list)
for row in test_all["en"]:
    if row["label"] in INTENTS:
        by_intent[row["label"]].append(row["id"])
test_ids = [i for lab in INTENTS for i in rng.sample(by_intent[lab], N_PER_INTENT)]
lookup = {lang: {r["id"]: r for r in test_all[lang]} for lang in ["en", "ar"]}
test = {lang: [lookup[lang][i] for i in test_ids] for lang in ["en", "ar"]}
pool = {lang: [r for r in train_all[lang] if r["label"] in INTENTS] for lang in ["en", "ar"]}

print({lang: len(rows) for lang, rows in test.items()}, "test requests;",
      {lang: len(rows) for lang, rows in pool.items()}, "training requests in the example pool")
pd.DataFrame({"English": [r["text"] for r in test["en"][::N_PER_INTENT]],
              "Arabic": [r["text"] for r in test["ar"][::N_PER_INTENT]],
              "intent": [r["label"] for r in test["en"][::N_PER_INTENT]]})

Many Arabic requests are in dialect rather than Modern Standard Arabic, which is harder for a small model.

## 2. The model and the scorer

The model is loaded in 16-bit on the GPU, with left padding so every prompt in a batch ends where generation starts. `ask` turns lists of chat messages into replies through the model's chat template, thinking off, greedy decoding.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
print(f"{MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

def ask(message_lists, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE, **sampling):
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
             for m in message_lists]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # similar lengths share a batch
    replies = [None] * len(texts)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=bool(sampling),
                                 pad_token_id=tok.pad_token_id, **sampling)
        for i, row in zip(idx, out[:, enc["input_ids"].shape[1]:]):
            replies[i] = tok.decode(row, skip_special_tokens=True).strip()
    return replies

A prompt is a function from one request to its chat messages. `score` runs it on the test requests and prints the accuracy, the replies with no valid intent name, and the most frequent answers. `RESULTS` keeps every score.

In [ ]:
def pick_intent(reply):
    reply = reply.lower()
    for lab in sorted(INTENTS, key=len, reverse=True):
        if lab in reply:
            return lab
    return None

RESULTS = {}

def score(name, build, langs=("en", "ar")):
    for lang in langs:
        t0 = time.time()
        replies = ask([build(r["text"]) for r in test[lang]])
        preds = [pick_intent(x) for x in replies]
        acc = np.mean([p == r["label"] for p, r in zip(preds, test[lang])])
        RESULTS[(name, lang)] = acc
        top = collections.Counter(preds).most_common(3)
        print(f"{name:32s} {lang}: accuracy {acc:.0%} | no valid intent: {preds.count(None):3d} | "
              f"most frequent: {top} | {time.time() - t0:.0f}s")

## 3. The baseline: label names only

The fixed rules go in the system message: the task, the 10 allowed intents and the output format. The request goes in the user message. No examples.

In [ ]:
NAMES_ONLY = ("Classify the user's request into exactly one of these intents:\n" + ", ".join(INTENTS) +
              "\nAnswer with the intent name only.")

def zero_shot(system):
    return lambda request: [{"role": "system", "content": system}, {"role": "user", "content": request}]

score("names only", zero_shot(NAMES_ONLY))

**What to notice.** The most frequent answers: a small model guessing what a label covers falls back on a few favourite labels. Two fixes follow: say what each label means, or show labelled examples.

## 4. Say what each label means

One plain line per intent, as you would brief a new colleague. Write them from the confusions above.

In [ ]:
DEFINITIONS = {
    "alarm_set": "set or change an alarm",
    "weather_query": "ask about the weather",
    "play_music": "play a song, an artist, a genre or a playlist",
    "calendar_set": "add an event or a reminder to the calendar",
    "news_query": "ask for the news",
    "email_sendemail": "write or send an email",
    "cooking_recipe": "ask how to cook something, or for a recipe",
    "transport_query": "ask about trains, buses, taxis, tickets or traffic",
    "datetime_query": "ask for the date or the time, here or in another place",
    "iot_hue_lightoff": "turn off the lights",
}
WITH_DEFINITIONS = ("Classify the user's request into exactly one of these intents:\n" +
                    "\n".join(f"- {k}: {v}" for k, v in DEFINITIONS.items()) +
                    "\nAnswer with the intent name only.")
print(len(tok(NAMES_ONLY)["input_ids"]), "tokens with names only;",
      len(tok(WITH_DEFINITIONS)["input_ids"]), "with definitions")
score("names with definitions", zero_shot(WITH_DEFINITIONS))

**What to notice.** Compare with the baseline in both languages. The definitions cost about a hundred tokens per request and need no labelled examples.

## 5. Examples: the same ones for everyone, or the most similar ones

Each example becomes an earlier user turn (the request) and assistant turn (the intent). Two ways of choosing the 10 examples:

- **one per intent**: the same 10 training requests for every test request, in random order
- **most similar**: for each test request, the 10 training requests that look most like it, the closest one last

Similarity is the cosine between TF-IDF vectors of character pieces (2 to 4 characters), which copes with Arabic spelling variants and needs no extra model.

In [ ]:
def turns(system, examples, request):
    messages = [{"role": "system", "content": system}]
    for e in examples:
        messages += [{"role": "user", "content": e["text"]}, {"role": "assistant", "content": e["label"]}]
    return messages + [{"role": "user", "content": request}]

similar = {}
for lang in ["en", "ar"]:
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
    pool_matrix = vec.fit_transform([r["text"] for r in pool[lang]])
    for r in test[lang]:
        sims = (vec.transform([r["text"]]) @ pool_matrix.T).toarray()[0]
        similar[r["text"]] = [pool[lang][j] for j in sims.argsort()[-N_EXAMPLES:]]   # most similar last

one_each = {lang: [rng.choice([r for r in pool[lang] if r["label"] == lab]) for lab in INTENTS] for lang in ["en", "ar"]}
for lang in one_each:
    rng.shuffle(one_each[lang])

request = test["ar"][0]["text"]
print("Request:", request, "| correct intent:", test["ar"][0]["label"])
for e in similar[request][-5:]:
    print("  neighbour:", e["text"], "->", e["label"])

Both example sets, scored per language.

In [ ]:
for lang in ["en", "ar"]:
    score("10 examples, one per intent", lambda q, lang=lang: turns(NAMES_ONLY, one_each[lang], q), langs=(lang,))
score("10 most similar examples", lambda q: turns(NAMES_ONLY, similar[q], q))

**What to notice.** When the neighbours contain the right label, phrased the same way, the model mostly has to copy. Retrieving examples is cheap and keeps the prompt the same length.

## 6. Wording: same task, different words

Only the wording of the instruction changes. The last variant is the first one written in Arabic, intent names kept in English.

In [ ]:
LIST = ", ".join(INTENTS)
WORDINGS = {
    "classify ... answer with the name": NAMES_ONLY,
    "which intent does it express?": f"Which of these intents does the request express? {LIST}. Reply with one intent name.",
    "Intents / Task / Output": (f"Intents: {' | '.join(INTENTS)}\nTask: label the user's request with one intent.\n"
                                "Output: the intent name, nothing else."),
    "smart-speaker framing": (f"You will receive a request that a user said to a smart speaker. Choose its intent "
                              f"from this list: {LIST}. Respond with exactly one intent from the list."),
    "terse": f"intent of the request? options: {LIST}",
    "the first one, in Arabic": "صنّف طلب المستخدم إلى نية واحدة فقط من هذه النوايا:\n" + LIST + "\nأجب باسم النية فقط.",
}
for name, system in WORDINGS.items():
    score(f"wording: {name}", zero_shot(system))

In [ ]:
labels = list(WORDINGS)
fig, ax = plt.subplots(figsize=(10, 3.5))
x = np.arange(len(labels))
for k, lang in enumerate(["en", "ar"]):
    ax.bar(x + (k - 0.5) * 0.38, [RESULTS[(f"wording: {n}", lang)] * 100 for n in labels], width=0.38, label=lang)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("accuracy (%)")
ax.set_title(f"Six wordings of one instruction, {MODEL_ID}")
ax.legend()
plt.tight_layout()
plt.show()

**What to notice.** The spread between the best and worst wording, and whether the best English wording is also best for Arabic. Try several wordings, and test again when the model changes.

## 7. A persona sets the audience

A persona changes who an answer is written for. We ask `N_QUESTIONS` questions from Natural Questions with no persona, as a primary-school teacher and as a university professor, and measure the Flesch-Kincaid reading grade:

$$\text{grade} = 0.39 \cdot \frac{\text{words}}{\text{sentences}} + 11.8 \cdot \frac{\text{syllables}}{\text{words}} - 15.59$$

Where the grade is roughly the school year a reader needs; syllables are counted as groups of vowels.

In [ ]:
questions = [q[0].upper() + q[1:] + "?" for q in
             load_dataset("google-research-datasets/nq_open", split="validation").select(range(N_QUESTIONS))["question"]]
PERSONAS = {"no persona": None,
            "primary-school teacher": "You are a primary-school teacher. You explain things to 10-year-old children: "
                                      "short sentences, everyday words, and an example from daily life.",
            "university professor": "You are a university professor. You answer for specialists in the field, "
                                    "with precise technical terms."}

def syllables(word):
    word = word.lower()
    n = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ee")) and n > 1:
        n -= 1
    return max(1, n)

def grade(text):
    sentences = [s for s in re.split(r"[.!?]+", text) if re.search(r"[A-Za-z]", s)]
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", text)
    if not words:
        return float("nan")
    return 0.39 * len(words) / max(1, len(sentences)) + 11.8 * sum(map(syllables, words)) / len(words) - 15.59

answers = {}
for name, system in PERSONAS.items():
    msgs = [([{"role": "system", "content": system}] if system else []) +
            [{"role": "user", "content": q + " Answer in 3 to 4 sentences."}] for q in questions]
    answers[name] = ask(msgs, max_new_tokens=200)
    print(f"{name:24s} mean reading grade {np.nanmean([grade(a) for a in answers[name]]):5.1f}")
print("\nQuestion:", questions[0])
for name in PERSONAS:
    print(f"\n[{name}] {answers[name][0][:300]}")

**What to notice.** The teacher persona brings the grade down and the professor pushes it up. Use a persona for audience and tone; for correctness, use definitions and examples.

## 8. A chain of prompts over a long text

A long text hides 50 sentences "The passcode for the X archive is NNNN." among movie reviews; the question asks for one. With one prompt over the whole text, a small model often misses passcodes in the middle. The chain asks the same question of every piece of about `CHUNK_TOKENS` tokens (NONE if absent) and lets code keep the answer.

In [ ]:
reviews = [re.sub(r"<[^>]+>", " ", r["text"]).strip()
           for r in load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=SEED).select(range(2000))]
NAMES = ["north", "south", "east", "west", "river", "harbour", "desert", "garden", "tower", "valley", "market",
         "bridge", "forest", "island", "palace", "museum", "station", "library", "castle", "canyon", "meadow",
         "summit", "lagoon", "orchard", "quarry", "plateau", "reef", "glacier", "oasis", "delta", "fjord", "grove",
         "hollow", "inlet", "jetty", "knoll", "ledge", "marsh", "nook", "outpost", "pier", "ridge", "shoal",
         "terrace", "upland", "vault", "wharf", "yard", "zenith", "cove"]
POSITIONS = {"first": 0, "middle": 25, "last": 49}

def build_doc(slot):
    order = NAMES[:]
    rng.shuffle(order)
    codes = {n: str(rng.randint(1000, 9999)) for n in order}
    hay, n_tok = [], 0
    while n_tok < DOC_TOKENS:
        p = rng.choice(reviews)
        hay.append(p)
        n_tok += len(tok(p)["input_ids"])
    paragraphs, step = [], len(hay) / len(order)
    for i, n in enumerate(order):                       # the 50 sentences spread evenly through the text
        paragraphs.append(f"The passcode for the {n} archive is {codes[n]}.")
        paragraphs.extend(hay[int(round(i * step)):int(round((i + 1) * step))])
    return paragraphs, order[slot], codes[order[slot]]

docs = [(pos, *build_doc(slot)) for pos, slot in POSITIONS.items() for _ in range(N_DOCS)]
question = lambda name: (f"What is the passcode for the {name} archive? Answer with the number only. "
                         "If the text does not give it, answer NONE.")
print(len(docs), "texts of about", DOC_TOKENS, "tokens")

First one prompt per text, then one prompt per piece. `pieces` cuts the text at paragraph boundaries.

In [ ]:
def pieces(paragraphs):
    out, cur, n = [], [], 0
    for p in paragraphs:
        m = len(tok(p)["input_ids"])
        if cur and n + m > CHUNK_TOKENS:
            out.append(cur)
            cur, n = [], 0
        cur.append(p)
        n += m
    return out + ([cur] if cur else [])

read = lambda paragraphs, name: [{"role": "user", "content": "Read the text below.\n\n" + "\n\n".join(paragraphs) +
                                  "\n\n" + question(name)}]
t0 = time.time()
direct = ask([read(p, name) for _, p, name, _ in docs], batch_size=2)
print(f"one prompt per text: {len(docs)} calls, {time.time() - t0:.0f}s")

t0 = time.time()
calls = [(d, read(piece, docs[d][2])) for d in range(len(docs)) for piece in pieces(docs[d][1])]
replies = ask([m for _, m in calls])
found = collections.defaultdict(list)
for (d, _), reply in zip(calls, replies):
    numbers = re.findall(r"\b\d{4}\b", reply)
    if numbers and "NONE" not in reply.upper():
        found[d].append(numbers[0])
chain = [collections.Counter(found[d]).most_common(1)[0][0] if found[d] else None for d in range(len(docs))]
print(f"one prompt per piece: {len(calls)} calls, {time.time() - t0:.0f}s")

table = pd.DataFrame({"position": [pos for pos, *_ in docs],
                      "one prompt": [code in reply for (_, _, _, code), reply in zip(docs, direct)],
                      "one prompt per piece": [c == code for (_, _, _, code), c in zip(docs, chain)]})
(table.groupby("position", sort=False).mean() * 100).round(0)

**What to notice.** Position matters for the single prompt and much less for the chain: no piece is long enough to have a middle. The price is many more calls. Read what each step returns before trusting the result.

## 9. All results

Every intent score from this notebook, side by side.

In [ ]:
rows = sorted({name for name, _ in RESULTS}, key=list(dict.fromkeys(n for n, _ in RESULTS)).index)
summary = pd.DataFrame({lang: [RESULTS.get((n, lang), np.nan) * 100 for n in rows] for lang in ["en", "ar"]}, index=rows)
summary.round(0)

## Summary

- A prompt is tested like code: the same labelled requests, greedy decoding, one score per version and language.
- One plain line per label, and examples chosen per request by similarity, are two cheap ways to tell a small model what each label means.
- Small wording changes move the score a lot, and the best wording differs between languages and models.
- A persona changes the audience and tone of an answer; a chain of short prompts handles a text too long for one prompt to use well.

## Exercises

1. Change `MODEL_ID` to `"Qwen/Qwen3-1.7B"` and rerun everything. Which changes still help, and which matter less with the larger model?
2. Rewrite the line in `DEFINITIONS` for the label the model confuses most, and score again. Then write the definitions in Arabic for the Arabic requests.
3. Count, for each test request, whether its correct intent is among its 10 most similar examples, and relate that count to the accuracy of section 5.
4. Change `CHUNK_TOKENS` to 250 and 1,000 in section 8. How do accuracy and the number of calls change?

## Further Reading

- [Brown et al., 2020. Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165)
- [Liu et al., 2021. What Makes Good In-Context Examples for GPT-3?](https://arxiv.org/abs/2101.06804)
- [Zhao et al., 2021. Calibrate Before Use: Improving Few-Shot Performance of Language Models](https://arxiv.org/abs/2102.09690)
- [Sclar et al., 2023. Quantifying Language Models' Sensitivity to Spurious Features in Prompt Design](https://arxiv.org/abs/2310.11324)
- [Zheng et al., 2023. When "A Helpful Assistant" Is Not Really Helpful: Personas in System Prompts Do Not Improve Performances of Large Language Models](https://arxiv.org/abs/2311.10054)
- [Liu et al., 2023. Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172)
- [FitzGerald et al., 2022. MASSIVE: A 1M-Example Multilingual Natural Language Understanding Dataset with 51 Typologically-Diverse Languages](https://arxiv.org/abs/2204.08582)

### Contributed by: Yazen Shaebi